In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
pd.set_option('display.max_columns', None)      
pd.set_option('display.float_format', '{:.4f}'.format)

In [3]:
#Bulk Load Datasets
DATA_DIR = r"E:\Portfolio_Projects\Supply_Chain_FMCG\data"  

dim_customers        = pd.read_csv(os.path.join(DATA_DIR, 'dim_customers.csv'))
dim_products         = pd.read_csv(os.path.join(DATA_DIR, 'dim_products.csv'))
dim_date             = pd.read_csv(os.path.join(DATA_DIR, 'dim_date.csv'))
dim_targets_orders   = pd.read_csv(os.path.join(DATA_DIR, 'dim_targets_orders.csv'))
fact_order_lines     = pd.read_csv(os.path.join(DATA_DIR, 'fact_order_lines.csv'))
fact_orders_agg      = pd.read_csv(os.path.join(DATA_DIR, 'fact_orders_aggregate.csv'))

In [4]:
#Shape Check
frames = {
    'dim_customers':      dim_customers,
    'dim_products':       dim_products,
    'dim_date':           dim_date,
    'dim_targets_orders': dim_targets_orders,
    'fact_order_lines':   fact_order_lines,
    'fact_orders_agg':    fact_orders_agg,
}

for name, df in frames.items():
    print(f"{name:30s}  rows: {df.shape[0]:>6,}   cols: {df.shape[1]}")

dim_customers                   rows:     35   cols: 3
dim_products                    rows:     18   cols: 3
dim_date                        rows:    183   cols: 3
dim_targets_orders              rows:     35   cols: 4
fact_order_lines                rows: 57,096   cols: 11
fact_orders_agg                 rows: 31,729   cols: 6


In [5]:
#Fetch column names and datatypes
for name, df in frames.items():
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(df.dtypes.to_string())


──────────────────────────────────────────────────
  dim_customers
──────────────────────────────────────────────────
customer_id       int64
customer_name    object
city             object

──────────────────────────────────────────────────
  dim_products
──────────────────────────────────────────────────
product_name    object
product_id       int64
category        object

──────────────────────────────────────────────────
  dim_date
──────────────────────────────────────────────────
date       object
mmm_yy     object
week_no    object

──────────────────────────────────────────────────
  dim_targets_orders
──────────────────────────────────────────────────
customer_id       int64
ontime_target%    int64
infull_target%    int64
otif_target%      int64

──────────────────────────────────────────────────
  fact_order_lines
──────────────────────────────────────────────────
order_id                object
order_placement_date    object
customer_id              int64
product_id         

In [6]:
#Renaming columns to snake_case for consistency. Source file uses mixed-case names with spaces (In Full, On Time) which break attribute access, and delivery_qty deviates from the metadata specification.
fact_order_lines.rename(columns={
    'delivery_qty':     'delivered_qty',
    'In Full':          'in_full_line',
    'On Time':          'on_time_line',
    'On Time In Full':  'otif_line'
}, inplace=True)

print("fact_order_lines columns after rename:")
print(fact_order_lines.columns.tolist())

fact_order_lines columns after rename:
['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty', 'agreed_delivery_date', 'actual_delivery_date', 'delivered_qty', 'in_full_line', 'on_time_line', 'otif_line']


In [7]:
#Parsing date columns to datetime64. Three different string formats exist across files and each requires its explicit format string. 
#Using format= prevents pandas from guessing incorrectly on ambiguous dates.

#dim_date - format: 01-Apr-22  (%d-%b-%y)
dim_date['date']   = pd.to_datetime(dim_date['date'],   format='%d-%b-%y')
dim_date['mmm_yy'] = pd.to_datetime(dim_date['mmm_yy'], format='%d-%b-%y')

#fact_order_lines - format: Tuesday, March 1, 2022  (%A, %B %d, %Y)
fact_order_lines['order_placement_date'] = pd.to_datetime(
    fact_order_lines['order_placement_date'], format='%A, %B %d, %Y')
fact_order_lines['agreed_delivery_date'] = pd.to_datetime(
    fact_order_lines['agreed_delivery_date'], format='%A, %B %d, %Y')
fact_order_lines['actual_delivery_date'] = pd.to_datetime(
    fact_order_lines['actual_delivery_date'], format='%A, %B %d, %Y')

#fact_orders_agg - format: 01-Mar-22  (%d-%b-%y)
fact_orders_agg['order_placement_date'] = pd.to_datetime(
    fact_orders_agg['order_placement_date'], format='%d-%b-%y')

In [8]:
#Verifying that all date columns successfully converted. 
#If any date column is still an object dtype here, that means there is a parse failure that would silently corrupt every time-based metric downstream.

print("\nDate dtype verification:")
print(f"  dim_date['date']                        : {dim_date['date'].dtype}")
print(f"  dim_date['mmm_yy']                      : {dim_date['mmm_yy'].dtype}")
print(f"  fact_order_lines['order_placement_date']: {fact_order_lines['order_placement_date'].dtype}")
print(f"  fact_order_lines['agreed_delivery_date']: {fact_order_lines['agreed_delivery_date'].dtype}")
print(f"  fact_order_lines['actual_delivery_date']: {fact_order_lines['actual_delivery_date'].dtype}")
print(f"  fact_orders_agg['order_placement_date'] : {fact_orders_agg['order_placement_date'].dtype}")


Date dtype verification:
  dim_date['date']                        : datetime64[ns]
  dim_date['mmm_yy']                      : datetime64[ns]
  fact_order_lines['order_placement_date']: datetime64[ns]
  fact_order_lines['agreed_delivery_date']: datetime64[ns]
  fact_order_lines['actual_delivery_date']: datetime64[ns]
  fact_orders_agg['order_placement_date'] : datetime64[ns]


In [9]:
#Checking for null values across all 6 tables. 
#Null values in foreign keys (customer_id, product_id) will silently drop records from joins and corrupt downstream metrics. 
#Must be zero in key columns before any analysis proceeds.

print("NULL COUNT PER COLUMN\n")
for name, df in frames.items():
    null_counts = df.isnull().sum()
    total_nulls = null_counts.sum()
    print(f"  {name} - total nulls: {total_nulls}")
    if total_nulls > 0:
        print(null_counts[null_counts > 0].to_string())
    print()

NULL COUNT PER COLUMN

  dim_customers - total nulls: 0

  dim_products - total nulls: 0

  dim_date - total nulls: 0

  dim_targets_orders - total nulls: 0

  fact_order_lines - total nulls: 0

  fact_orders_agg - total nulls: 0



In [10]:
#Checking for duplicate rows in both fact tables. 
#A duplicate row in fact_order_lines artificially increases order line counts and fill rate metrics. 
#A duplicate row in fact_orders_agg double-counts orders, directly corrupting OT%, IF%, and OTIF%.

print("DUPLICATE ROW CHECK\n")

dup_lines = fact_order_lines.duplicated().sum()
dup_agg   = fact_orders_agg.duplicated().sum()

print(f"  fact_order_lines - duplicate rows : {dup_lines:,}")
print(f"  fact_orders_agg  - duplicate rows : {dup_agg:,}")

DUPLICATE ROW CHECK

  fact_order_lines - duplicate rows : 0
  fact_orders_agg  - duplicate rows : 0


In [11]:
#Checking specifically for duplicate order_id values in fact_orders_agg. 
#Each order_id must appear exactly once at the aggregate level and if the same order appears twice, every order-level service metric is overstated.

dup_order_ids = fact_orders_agg['order_id'].duplicated().sum()
print(f"  fact_orders_agg  - duplicate order_ids: {dup_order_ids:,}")

  fact_orders_agg  - duplicate order_ids: 0


#### No null values or duplicate records detected across any table. This confirms the dataset is structurally sound and all metrics computed downstream reflect actual operational performance rather than data quality artifacts.

In [12]:
#Checking unique value counts in categorical columns. 
#An unexpected extra value in city or category indicates a data entry error, a typo or inconsistent casing that would silently create a phantom group in every city or category breakdown.

print("UNIQUE VALUE COUNTS — CATEGORICAL COLUMNS\n")

print(f"  dim_customers  - city          : {dim_customers['city'].nunique()} unique | {dim_customers['city'].unique()}")
print(f"  dim_products   - category      : {dim_products['category'].nunique()} unique | {dim_products['category'].unique()}")
print(f"  dim_date       - week_no       : {dim_date['week_no'].nunique()} unique weeks")
print(f"  fact_order_lines - order_id    : {fact_order_lines['order_id'].nunique()} unique orders")
print(f"  fact_orders_agg  - order_id    : {fact_orders_agg['order_id'].nunique()} unique orders")

UNIQUE VALUE COUNTS — CATEGORICAL COLUMNS

  dim_customers  - city          : 3 unique | ['Surat' 'Ahmedabad' 'Vadodara']
  dim_products   - category      : 3 unique | ['Dairy' 'Food' 'beverages']
  dim_date       - week_no       : 27 unique weeks
  fact_order_lines - order_id    : 31729 unique orders
  fact_orders_agg  - order_id    : 31729 unique orders


In [14]:
#Checking that binary flag columns contain only 0 and 1, and that quantity columns contain no zero or negative values. 
#A corrupted flag silently breaks every service level metric built on top of it. A zero or negative quantity suggests a data entry or system error.

print("\nNUMERIC RANGE CHECKS\n")

# Flag columns must be 0 or 1 only
for col in ['in_full_line', 'on_time_line', 'otif_line']:
    vals = fact_order_lines[col].unique()
    print(f"  fact_order_lines['{col}'] unique values: {sorted(vals)}")

for col in ['on_time', 'in_full', 'otif']:
    vals = fact_orders_agg[col].unique()
    print(f"  fact_orders_agg['{col}']  unique values: {sorted(vals)}")

# Quantity columns must be positive
print(f"\n  fact_order_lines - order_qty    min: {fact_order_lines['order_qty'].min()}   max: {fact_order_lines['order_qty'].max()}")
print(f"  fact_order_lines - delivered_qty min: {fact_order_lines['delivered_qty'].min()}   max: {fact_order_lines['delivered_qty'].max()}")

# Target columns range check
print(f"\n  ontime_target%  range: {dim_targets_orders['ontime_target%'].min()} – {dim_targets_orders['ontime_target%'].max()}")
print(f"  infull_target%  range: {dim_targets_orders['infull_target%'].min()} – {dim_targets_orders['infull_target%'].max()}")
print(f"  otif_target%    range: {dim_targets_orders['otif_target%'].min()} – {dim_targets_orders['otif_target%'].max()}")


NUMERIC RANGE CHECKS

  fact_order_lines['in_full_line'] unique values: [np.int64(0), np.int64(1)]
  fact_order_lines['on_time_line'] unique values: [np.int64(0), np.int64(1)]
  fact_order_lines['otif_line'] unique values: [np.int64(0), np.int64(1)]
  fact_orders_agg['on_time']  unique values: [np.int64(0), np.int64(1)]
  fact_orders_agg['in_full']  unique values: [np.int64(0), np.int64(1)]
  fact_orders_agg['otif']  unique values: [np.int64(0), np.int64(1)]

  fact_order_lines - order_qty    min: 20   max: 500
  fact_order_lines - delivered_qty min: 16   max: 500

  ontime_target%  range: 75 – 92
  infull_target%  range: 65 – 82
  otif_target%    range: 49 – 75


In [15]:
#Validating customer_id across fact_order_lines and fact_orders_agg against dim_customers. 
#An orphan record in either fact table means that order silently disappears from every customer-level join, making that customer's service level invisible in every metric downstream.

valid_customer_ids = set(dim_customers['customer_id'])

fol_invalid_cust = fact_order_lines[
    ~fact_order_lines['customer_id'].isin(valid_customer_ids)
]
agg_invalid_cust = fact_orders_agg[
    ~fact_orders_agg['customer_id'].isin(valid_customer_ids)
]

print("FOREIGN KEY — customer_id\n")
print(f"  fact_order_lines - orphan customer_ids : {len(fol_invalid_cust):,}")
print(f"  fact_orders_agg  - orphan customer_ids : {len(agg_invalid_cust):,}")

FOREIGN KEY — customer_id

  fact_order_lines - orphan customer_ids : 0
  fact_orders_agg  - orphan customer_ids : 0


In [16]:
# Validating product_id across fact_order_lines and dim_products.
# Orphan records or unmatched IDs will cause missing product details during joins, leading to misleading insights downstream.

valid_product_ids = set(dim_products['product_id'])

fol_invalid_prod = fact_order_lines[
    ~fact_order_lines['product_id'].isin(valid_product_ids)
]

print("\nFOREIGN KEY - product_id\n")
print(f"  fact_order_lines - orphan product_ids  : {len(fol_invalid_prod):,}")


FOREIGN KEY - product_id

  fact_order_lines - orphan product_ids  : 0


In [17]:
# Validating date range alignment across all tables.
# Data in every table must fall within the identical date window. A variance in any table indicates an input mismatch.

print("\nDATE RANGE VALIDATION\n")

print(f"  dim_date range                : {dim_date['date'].min().date()} - {dim_date['date'].max().date()}")
print(f"  fact_order_lines placement    : {fact_order_lines['order_placement_date'].min().date()} - {fact_order_lines['order_placement_date'].max().date()}")
print(f"  fact_order_lines agreed       : {fact_order_lines['agreed_delivery_date'].min().date()} - {fact_order_lines['agreed_delivery_date'].max().date()}")
print(f"  fact_order_lines actual       : {fact_order_lines['actual_delivery_date'].min().date()} - {fact_order_lines['actual_delivery_date'].max().date()}")
print(f"  fact_orders_agg  placement    : {fact_orders_agg['order_placement_date'].min().date()} - {fact_orders_agg['order_placement_date'].max().date()}")


DATE RANGE VALIDATION

  dim_date range                : 2022-03-01 - 2022-08-30
  fact_order_lines placement    : 2022-03-01 - 2022-08-30
  fact_order_lines agreed       : 2022-03-02 - 2022-08-31
  fact_order_lines actual       : 2022-03-01 - 2022-09-03
  fact_orders_agg  placement    : 2022-03-01 - 2022-08-30


#### Actual_delivery_date extends to September 3, 2022 i.e., four days beyond the official data window of August 30. This is not a data error. These are late deliveries where AtliQ committed to August delivery but fulfilled in September. All delivery outcomes are fully captured in the dataset, confirming OT% calculations will reflect complete operational performance across the entire window.

In [18]:
#Verifying whether otif_target% equals ontime_target% × infull_target% across all 35 customers. 
#If true, AtliQ's targets assume OT and IF failures are statistically independent events and a significant business assumption that understates risk when both failures occur together on the same order.

dim_targets_orders['otif_calculated'] = (
    (dim_targets_orders['ontime_target%'] / 100) *
    (dim_targets_orders['infull_target%'] / 100) *
    100
).round(2)

dim_targets_orders['otif_match'] = (
    dim_targets_orders['otif_calculated'].round(0).astype(int) ==
    dim_targets_orders['otif_target%']
)

match_count    = dim_targets_orders['otif_match'].sum()
mismatch_count = (~dim_targets_orders['otif_match']).sum()

print("OTIF TARGET INDEPENDENCE CHECK\n")
print(f"  Rows where otif_target% = OT% × IF%  : {match_count}")
print(f"  Rows where formula does NOT hold      : {mismatch_count}")
print(f"\n  Sample of 5 rows:")
print(dim_targets_orders[
    ['customer_id','ontime_target%','infull_target%','otif_target%','otif_calculated','otif_match']
].head())

OTIF TARGET INDEPENDENCE CHECK

  Rows where otif_target% = OT% × IF%  : 35
  Rows where formula does NOT hold      : 0

  Sample of 5 rows:
   customer_id  ontime_target%  infull_target%  otif_target%  otif_calculated  \
0       789201              87              81            70          70.4700   
1       789202              85              81            69          68.8500   
2       789203              92              76            70          69.9200   
3       789301              89              78            69          69.4200   
4       789303              88              78            69          68.6400   

   otif_match  
0        True  
1        True  
2        True  
3        True  
4        True  


In [19]:
#Reconstructing order-level flags from line-level data by taking the minimum flag value per order that means an order is on time only if every line is on time. 
#Comparing reconstructed flags against fact_orders_agg to verify the aggregate table is a faithful summary of the line-level data. 
#Any mismatch indicates a data integrity issue that corrupts every service level metric downstream.

reconstructed = fact_order_lines.groupby('order_id').agg(
    on_time_reconstructed = ('on_time_line', 'min'),
    in_full_reconstructed = ('in_full_line', 'min'),
    otif_reconstructed    = ('otif_line',    'min')
).reset_index()

merged = fact_orders_agg.merge(reconstructed, on='order_id', how='inner')

merged['ot_match']   = merged['on_time'] == merged['on_time_reconstructed']
merged['if_match']   = merged['in_full'] == merged['in_full_reconstructed']
merged['otif_match'] = merged['otif']    == merged['otif_reconstructed']

print("AGGREGATE RECONCILIATION\n")
print(f"  Orders compared                        : {len(merged):,}")
print(f"  on_time  matches                       : {merged['ot_match'].sum():,}")
print(f"  in_full  matches                       : {merged['if_match'].sum():,}")
print(f"  otif     matches                       : {merged['otif_match'].sum():,}")
print(f"\n  on_time  mismatches                    : {(~merged['ot_match']).sum():,}")
print(f"  in_full  mismatches                    : {(~merged['if_match']).sum():,}")
print(f"  otif     mismatches                    : {(~merged['otif_match']).sum():,}")

AGGREGATE RECONCILIATION

  Orders compared                        : 31,729
  on_time  matches                       : 31,729
  in_full  matches                       : 31,729
  otif     matches                       : 31,729

  on_time  mismatches                    : 0
  in_full  mismatches                    : 0
  otif     mismatches                    : 0


#### Consolidating all data quality findings from Sections 1–7 into a single summary report. This documents what was checked, what was found, and what requires action before metric computation begins.

In [1]:
#Data Quality Summary Report

print("=" * 60)
print("  DATA QUALITY SUMMARY REPORT — AtliQ Mart OTIF Project")
print("=" * 60)

print("""
STRUCTURE & LOAD
  ✓ All 6 CSVs loaded with expected row counts
  ✓ No null values detected in any column across all tables
  ✓ No duplicate rows in fact_order_lines or fact_orders_agg
  ✓ No duplicate order_ids in fact_orders_agg

COLUMN STANDARDIZATION
  ✓ delivery_qty renamed to delivered_qty (metadata alignment)
  ✓ Line-level flags renamed: in_full_line, on_time_line, otif_line
    (disambiguated from order-level flags in fact_orders_agg)

DATE PARSING
  ✓ All 6 date columns parsed to datetime64[ns]
  ✓ Three source formats handled explicitly:
      dim_date          → %d-%b-%y
      fact_order_lines  → %A, %B %d, %Y
      fact_orders_agg   → %d-%b-%y

DATA QUALITY FINDINGS — ACTION REQUIRED
  ⚠ 'beverages' in dim_products.category uses lowercase
    All other categories use Title Case (Dairy, Food)
    Action: Standardize to 'Beverages' in 02_cleaning.ipynb

RANGE & INTEGRITY CHECKS
  ✓ City values: exactly 3 (Surat, Ahmedabad, Vadodara)
  ✓ All binary flags contain only [0, 1]
  ✓ delivered_qty minimum = 16 (no zero deliveries recorded)
  ✓ No orphan customer_ids or product_ids in any fact table

DATE RANGE FINDING
  ✓ Data window: 2022-03-01 to 2022-08-30 (6 months)
  ✓ actual_delivery_date extends to 2022-09-03
    This is expected — late deliveries fulfilled after window close
    All delivery outcomes are fully captured

BUSINESS RULE VALIDATION
  ✓ OTIF target = OT target × IF target confirmed across all 35
    customers. AtliQ's target-setting assumes statistical
    independence of OT and IF failures.
    Finding: This assumption underestimates risk when both
    failures occur simultaneously on the same order.
  ✓ fact_orders_agg reconciled against fact_order_lines
    All 31,729 orders match perfectly across on_time,
    in_full, and otif flags.

CONCLUSION
  Dataset is structurally sound and business-rule validated.
  One casing correction required before analysis proceeds.
  All metrics computed in subsequent notebooks are built on
  a verified, trustworthy foundation.
""")
print("=" * 60)

  DATA QUALITY SUMMARY REPORT — AtliQ Mart OTIF Project

STRUCTURE & LOAD
  ✓ All 6 CSVs loaded with expected row counts
  ✓ No null values detected in any column across all tables
  ✓ No duplicate rows in fact_order_lines or fact_orders_agg
  ✓ No duplicate order_ids in fact_orders_agg

COLUMN STANDARDIZATION
  ✓ delivery_qty renamed to delivered_qty (metadata alignment)
  ✓ Line-level flags renamed: in_full_line, on_time_line, otif_line
    (disambiguated from order-level flags in fact_orders_agg)

DATE PARSING
  ✓ All 6 date columns parsed to datetime64[ns]
  ✓ Three source formats handled explicitly:
      dim_date          → %d-%b-%y
      fact_order_lines  → %A, %B %d, %Y
      fact_orders_agg   → %d-%b-%y

DATA QUALITY FINDINGS — ACTION REQUIRED
  ⚠ 'beverages' in dim_products.category uses lowercase
    All other categories use Title Case (Dairy, Food)
    Action: Standardize to 'Beverages' in 02_cleaning.ipynb

RANGE & INTEGRITY CHECKS
  ✓ City values: exactly 3 (Surat, Ahmeda